In [73]:
# Project Setup
import pandas as pd
import numpy as np
from pathlib import Path



In [74]:
#Load Raw Data
try:
    ab_df = pd.read_csv(r"D:\2.DAP391m_Project\Ecommerce AB Testing 2022 Dataset1\ecommerce_ab_testing_2022_dataset1\ab_data.csv")
    country_df = pd.read_csv(r"D:\2.DAP391m_Project\Ecommerce AB Testing 2022 Dataset1\ecommerce_ab_testing_2022_dataset1\countries.csv")
    print("Import file csv successfully!")
except Exception:
    print(e)
    print("Can not import file CSV!")

Import file csv successfully!


In [75]:
#Basic Data Inspection
ab_df.shape
country_df.shape

ab_df.head()
country_df.head()

ab_df.info()
country_df.info()

ab_df.columns
country_df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294480 entries, 0 to 294479
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294480 non-null  int64 
 1   timestamp     294480 non-null  object
 2   group         294480 non-null  object
 3   landing_page  294480 non-null  object
 4   converted     294480 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290586 entries, 0 to 290585
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   user_id  290586 non-null  int64 
 1   country  290586 non-null  object
dtypes: int64(1), object(1)
memory usage: 4.4+ MB


Index(['user_id', 'country'], dtype='object')

In [76]:
#Missing Value Check
print(ab_df.isnull().sum())
print(country_df.isnull().sum())

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64
user_id    0
country    0
dtype: int64


In [77]:
# Duplicate User Check
ab_df["user_id"].duplicated().sum()



np.int64(3895)

In [78]:
country_df["user_id"].duplicated().sum()

np.int64(1)

In [79]:
#Drop duplicate user 
ab_df_cleaned = ab_df.drop_duplicates(subset="user_id")
country_df_cleaned = country_df.drop_duplicates(subset="user_id")

In [80]:
# A/B Assignment-Page Consistency Check
pd.crosstab(ab_df_cleaned["landing_page"], ab_df_cleaned["group"])


group,control,treatment
landing_page,,
new_page,1006,144315
old_page,144226,1038


In [81]:
# Remove Mismatch Records
ab_cleaned = ab_df_cleaned[
    ((ab_df_cleaned["group"] == "control") & (ab_df_cleaned["landing_page"] == "old_page")) |
    ((ab_df_cleaned["group"] == "treatment") & (ab_df_cleaned["landing_page"] == "new_page"))
].copy()

In [82]:
# Merge Country Data
df_merged = ab_cleaned.merge(
    country_df,
    on="user_id",
    how="left"
)
df_merged = df_merged.drop_duplicates(subset="user_id")

In [83]:
# Check after merge
print(df_merged.shape)
print(df_merged.isnull().sum())
print(df_merged.head())

(288541, 6)
user_id         0
timestamp       0
group           0
landing_page    0
converted       0
country         0
dtype: int64
   user_id timestamp      group landing_page  converted country
0   851104   11:48.6    control     old_page          0      US
1   804228   01:45.2    control     old_page          0      US
2   661590   55:06.2  treatment     new_page          0      US
3   853541   28:03.1  treatment     new_page          0      US
4   864975   52:26.2    control     old_page          1      US


In [84]:
# Final Clean Data Check
df_merged.shape
df_merged.info()
df_merged.isnull().sum()
df_merged.head()

pd.crosstab(df_merged["landing_page"], df_merged["group"])
df_merged["user_id"].duplicated().sum()


<class 'pandas.core.frame.DataFrame'>
Index: 288541 entries, 0 to 288541
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       288541 non-null  int64 
 1   timestamp     288541 non-null  object
 2   group         288541 non-null  object
 3   landing_page  288541 non-null  object
 4   converted     288541 non-null  int64 
 5   country       288541 non-null  object
dtypes: int64(2), object(4)
memory usage: 15.4+ MB


np.int64(0)

In [85]:
pd.crosstab(df_merged["landing_page"], df_merged["group"])


group,control,treatment
landing_page,,
new_page,0,144315
old_page,144226,0


In [86]:
df_merged.to_csv(
    r"D:\2.DAP391m_Project\dataset_cleaned\dataset_cleaned.csv",
    index=False
)